In [1]:
!nvidia-smi
!pip -q install -U sentence-transformers datasets accelerate rank_bm25
import torch, sentence_transformers, transformers
print("GPU:", torch.cuda.is_available(), "| ST", sentence_transformers.__version__, "| HF", transformers.__version__)

Fri Oct 16 14:29:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/esci"
for d in ["data", "dataset_v2.3", "models", "results"]:
    os.makedirs(f"{BASE}/{d}", exist_ok=True)
print(sorted(os.listdir(BASE)))

Mounted at /content/drive
['data', 'dataset_v2.3', 'models', 'results']


In [3]:
import os
BASE = "/content/drive/MyDrive/esci"
RAW = f"{BASE}/data"
URL = "https://github.com/amazon-science/esci-data/raw/main/shopping_queries_dataset/"
FILES = ["shopping_queries_dataset_examples.parquet",
         "shopping_queries_dataset_products.parquet",
         "shopping_queries_dataset_sources.csv"]

for f in FILES:
    p = f"{RAW}/{f}"
    if not (os.path.exists(p) and os.path.getsize(p) > 100_000):
        !wget -q -L -O "{p}" "{URL}{f}"
    print(f, "->", round(os.path.getsize(p)/1e6, 1), "MB")

shopping_queries_dataset_examples.parquet -> 51.3 MB
shopping_queries_dataset_products.parquet -> 1108.9 MB
shopping_queries_dataset_sources.csv -> 1.7 MB


In [4]:
import pandas as pd, pyarrow.parquet as pq
RAW = "/content/drive/MyDrive/esci/data"

ex = pd.read_parquet(f"{RAW}/shopping_queries_dataset_examples.parquet")
src = pd.read_csv(f"{RAW}/shopping_queries_dataset_sources.csv")
pm = pq.ParquetFile(f"{RAW}/shopping_queries_dataset_products.parquet").metadata

print("examples:", ex.shape)
print("locales:", ex["product_locale"].value_counts().to_dict())
print("US queries:", ex.loc[ex.product_locale=="us", "query_id"].nunique())   # expect 97,345
print("products rows:", pm.num_rows)
print("sources:", src.shape)

examples: (2621288, 9)
locales: {'us': 1818825, 'jp': 446053, 'es': 356410}
US queries: 97345
products rows: 1814924
sources: (130652, 2)
